# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [94]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [95]:
# Load the dataset from Phase 2 (update the path as needed)
DATA_PATH = "../data/raw/uber-raw-data-apr14.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 564516 rows x 4 columns


,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [96]:
# TODO: Select the relevant columns and rows for your analysis.
# Document your rationale for including or excluding specific features.

# Columns to keep (Base removed)
columns_to_keep = ['Date/Time', 'Lat', 'Lon']

# Columns to drop
columns_to_drop = ['Base']

# Reasons for dropping
drop_reason = {
    'Base': 'Internal operational identifier with no explicit geographic meaning or direct relevance to spatial-temporal analysis goals'
}

# Select data
df_selected = df[columns_to_keep].copy()

print(f"Shape after column selection: {df_selected.shape}")

Shape after column selection: (564516, 3)


In [97]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

# df_selected = df_selected[df_selected['some_column'].notna()]
# print(f"Shape after row selection: {df_selected.shape}")

---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [98]:
# TODO: Handle missing values.
df_clean = df_selected.copy()

# No missing values detected, so no imputation needed
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")

Missing values remaining: 0


In [99]:
# TODO: Remove duplicate records.
df_clean = df_selected.copy()

before = len(df_clean)

df_clean = df_clean.drop_duplicates()

after = len(df_clean)

print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")

Removed 7943 duplicate rows. Remaining: 556573 rows.


In [100]:
# TODO: Handle outliers (geographical filtering)

df_clean = df_clean[
    (df_clean['Lat'] >= 40.5) & (df_clean['Lat'] <= 41.0) &
    (df_clean['Lon'] >= -74.5) & (df_clean['Lon'] <= -73.5)
]

print(f"Remaining rows after outlier removal: {len(df_clean)}")

Remaining rows after outlier removal: 555752


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [101]:
# TODO: Create derived attributes / new features.

# Ensure datetime format
df_clean['Date/Time'] = pd.to_datetime(df_clean['Date/Time'])

# Extract time-based features
df_clean['hour'] = df_clean['Date/Time'].dt.hour
df_clean['day'] = df_clean['Date/Time'].dt.day
df_clean['weekday'] = df_clean['Date/Time'].dt.weekday
df_clean['month'] = df_clean['Date/Time'].dt.month

# Create simple time period categories
df_clean['time_period'] = pd.cut(
    df_clean['hour'],
    bins=[-1, 6, 12, 18, 24],
    labels=['Night', 'Morning', 'Afternoon', 'Evening']
)

print(df_clean.head())

            Date/Time      Lat      Lon  hour  day  weekday  month time_period
0 2014-04-01 00:11:00  40.7690 -73.9549     0    1        1      4       Night
1 2014-04-01 00:17:00  40.7267 -74.0345     0    1        1      4       Night
2 2014-04-01 00:21:00  40.7316 -73.9873     0    1        1      4       Night
3 2014-04-01 00:28:00  40.7588 -73.9776     0    1        1      4       Night
4 2014-04-01 00:33:00  40.7594 -73.9722     0    1        1      4       Night


In [102]:
# TODO: Encode categorical variables.

# No categorical variables remain after feature engineering and removal of 'Base'
print("No categorical variables to encode in the current dataset.")

No categorical variables to encode in the current dataset.


In [103]:
# TODO: Scale / normalise numerical features if required.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

cols_to_scale = ['Lat', 'Lon', 'hour', 'weekday']

df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])

print("Scaling complete.")

Scaling complete.


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [104]:
# TODO: Integrate data from multiple sources (if applicable).

integration_note = """
This project uses a single dataset (Uber pickup data for April 2014).
Therefore, no data integration (merging or concatenation) was required.
All analysis is performed on a unified dataset without multiple sources.
"""

print(integration_note)


This project uses a single dataset (Uber pickup data for April 2014).
Therefore, no data integration (merging or concatenation) was required.
All analysis is performed on a unified dataset without multiple sources.



In [105]:
# Optional: Verify the integrated data
# df_integrated.head()

---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [106]:
# TODO: Apply final formatting — data types, column order, renaming.

# Ensure datetime format
df_clean['Date/Time'] = pd.to_datetime(df_clean['Date/Time'])

# rename columns for clarity
df_clean = df_clean.rename(columns={
    'Date/Time': 'datetime',
    'Lat': 'latitude',
    'Lon': 'longitude'
})

# Reorder columns logically
df_final = df_clean[[
    'datetime',
    'latitude',
    'longitude',
    'hour',
    'day',
    'weekday',
    'month',
    'time_period'
]]

In [107]:
# TODO: Verify the final prepared dataset.

print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)

print(f"Shape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicates: {df_final.duplicated().sum()}")

print("\nColumn types:")
print(df_final.dtypes)

df_final.head()

FINAL PREPARED DATASET SUMMARY
Shape: (555752, 8)
Missing values: 0
Duplicates: 0

Column types:
datetime       datetime64[ns]
latitude              float64
longitude             float64
hour                  float64
day                     int32
weekday               float64
month                   int32
time_period          category
dtype: object


,datetime,latitude,longitude,hour,day,weekday,month,time_period
0,2014-04-01 00:11:00,0.858177,0.467912,-2.464574,1,-1.025062,4,Night
1,2014-04-01 00:17:00,-0.379095,-1.204160,-2.464574,1,-1.025062,4,Night
2,2014-04-01 00:21:00,-0.235770,-0.212680,-2.464574,1,-1.025062,4,Night
3,2014-04-01 00:28:00,0.559828,-0.008923,-2.464574,1,-1.025062,4,Night
4,2014-04-01 00:33:00,0.577378,0.104509,-2.464574,1,-1.025062,4,Night


In [108]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).

OUTPUT_PATH = "../data/processed/uber_prepared_data.csv"

df_final.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

Saved to: ../data/processed/uber_prepared_data.csv
